# 01 - Data prep (Colab)

Downloads VTaC, **verifies the schema** (fill `docs/data.md` -> Findings from the printed output /
`reports/data_survey.json`), preprocesses every event with the shared `vtac` package, and writes:

- `processed/windows_off{offset}.npz` - model inputs for each window end-offset (default +2.0 s and
  the official-benchmark 0.0 s, for ablation W0 vs W1)
- `replay/replay_sample.npz` + `replay_index.json` - ~25 **test** alarms for the laptop
- `reports/data_survey.json`, `processed/windows_meta.json`

All preprocessing comes from `vtac` (never re-implemented here). Simulated real-time replay only;
no live monitor integration.

> Status: written but **not yet executed on Colab**. Expect to fix small things on the first run
> (e.g. split value names, channel names) - that is what the verification cells are for.

In [ ]:
# ---- Setup: mount Drive, install vtac from GitHub (pinned to a ref) -------------------------
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

VTAC_REF = "main"  # replace with a tag/commit once stable (reproducibility!)
!pip -q install "vtac[train] @ git+https://github.com/siddhu-exe/vtac-alarm-monitor.git@{VTAC_REF}"

import vtac
from vtac.provenance import installed_git_ref
GIT_REF = installed_git_ref(VTAC_REF)
print("vtac", vtac.__version__, "ref", GIT_REF)

DRIVE = Path("/content/drive/MyDrive/vtac")
for sub in ("processed", "replay", "models", "reports"):
    (DRIVE / sub).mkdir(parents=True, exist_ok=True)
SEED = 42

## 1. Download VTaC (open access, PhysioNet)

In [ ]:
RAW = Path("/content/vtac_raw")  # Colab local disk (fast); ~4 GB
if not (RAW / "event_labels.csv").exists():
    !wget -r -N -c -np -nH --cut-dirs=3 -q -R "index.html*" -P {RAW} https://physionet.org/files/vtac/1.0/
!ls {RAW} | head
!ls {RAW}/waveforms | head -3
!du -sh {RAW}

## 2. Labels + official split: verify before trusting

Checks: counts (5,037 events, 1,441 true), 1:1 join, class balance per split, **patient-level split**
(`record` = patient; a patient can have up to 5 events). Stops if the split leaks patients.

In [ ]:
import numpy as np, pandas as pd, json
from vtac.config import VtacConfig
from vtac.io import load_labels, load_split
from vtac.splits import check_patient_level, add_validation_split

cfg = VtacConfig()
labels = load_labels(RAW / "event_labels.csv")
split = load_split(RAW / "benchmark_data_split.csv")
print("labels", labels.shape, "| true:", int(labels.y.sum()), f"({labels.y.mean():.2%})")
print("split ", split.shape, "| split values:", split.split.value_counts().to_dict())
assert labels.event.is_unique and split.event.is_unique

df = split.merge(labels[["event", "y"]], on="event", how="outer", validate="1:1", indicator=True)
print(df._merge.value_counts().to_dict())
assert (df._merge == "both").all(), "labels and split disagree on the event set"
df = df.drop(columns="_merge")

print(df.groupby("split").y.agg(n="size", n_true="sum", true_rate="mean"))
report = check_patient_level(df)
print({k: v for k, v in report.items() if k != "leaking_patients"})
assert report["patient_level"], f"split leaks patients: {report['leaking_patients'][:10]}"

In [ ]:
# If the official split has no validation set, carve one from TRAIN (grouped by patient,
# stratified by label, fixed seed). Test is never touched.
if not ({"val", "valid", "validation"} & set(df.split.str.lower())):
    df = add_validation_split(df, val_frac=0.15, seed=SEED)
    print("carved validation split")
print(df.groupby("split").y.agg(n="size", n_true="sum", true_rate="mean"))
assert check_patient_level(df)["patient_level"]
assert {"train", "val", "test"} <= set(df.split), f"unexpected split names: {set(df.split)}"

## 3. Survey every header (no signal loading): fs, length, channel names, units

Copy the printed summary into `docs/data.md` -> Findings, and into `vtac/config.py` if anything
differs from the defaults (channel names, ECG priority, fs, segment length).

In [ ]:
import wfdb
from collections import Counter
from vtac.channels import select_slots

survey = []
for rec, ev in zip(df.record, df.event):
    h = wfdb.rdheader(str(RAW / "waveforms" / rec / ev))
    survey.append(dict(event=ev, fs=h.fs, sig_len=h.sig_len, names=list(h.sig_name), units=list(h.units)))

print("fs:", Counter(s["fs"] for s in survey))
print("sig_len:", Counter(s["sig_len"] for s in survey))
name_counts = Counter(n for s in survey for n in s["names"])
print("channel names:", dict(name_counts))
print("channel sets:", Counter(tuple(sorted(s["names"])) for s in survey).most_common(15))
units = {}
for s in survey:
    for n, u in zip(s["names"], s["units"]):
        units.setdefault(n, Counter())[u] += 1
print("units:", {n: dict(c) for n, c in units.items()})

slot_pat = Counter(tuple(k for k, v in select_slots(s["names"], cfg).items() if v is None) for s in survey)
print("MISSING slot patterns (per cfg.ecg_priority/ppg_names/abp_names):", slot_pat)
no_ecg2 = sum(1 for s in survey if select_slots(s["names"], cfg)["ECG_2"] is None)
no_puls = sum(1 for s in survey if all(select_slots(s["names"], cfg)[k] is None for k in ("PPG", "ABP")))
print("events with <2 ECG:", no_ecg2, "| with no PPG and no ABP:", no_puls)
(DRIVE / "reports" / "data_survey.json").write_text(json.dumps(dict(
    fs=dict(Counter(s["fs"] for s in survey)), sig_len=dict(Counter(s["sig_len"] for s in survey)),
    channel_names=dict(name_counts), missing_slot_patterns={",".join(k) or "none": v for k, v in slot_pat.items()},
    units={n: dict(c) for n, c in units.items()}), indent=2))

**Stop and check** before continuing: every `fs` is 250, every `sig_len` is 90000, and channel
names are all covered by `cfg` (a name that falls into the 'rest' ECG fallback or is silently
dropped needs adding to `VtacConfig`). If not, update `src/vtac/config.py`, push, reinstall.

## 4. Preprocess every event (via `vtac`) for each window offset

`OFFSETS = [2.0, 0.0]`: +2.0 s is the project default (D18); 0.0 is the official benchmark window
(samples 72500:75000) for ablation W0. Also picks the ~25-alarm **test** replay sample (10 true +
15 false, fixed seed) and keeps its RAW slot-ordered segments.

In [ ]:
from tqdm.auto import tqdm
from vtac.io import load_segment
from vtac.preprocess import to_slot_array, preprocess_slots
from vtac.dataset import save_windows, save_replay

OFFSETS = [2.0, 0.0]
cfgs = {o: cfg.with_updates(window_end_offset_s=o, vtac_git_ref=GIT_REF) for o in OFFSETS}

rng = np.random.default_rng(SEED)
test = df[df.split == "test"]
sample_ids = set(rng.choice(test[test.y == 1].event.to_numpy(), 10, replace=False)) | \
             set(rng.choice(test[test.y == 0].event.to_numpy(), 15, replace=False))

W = {o: {"signals": [], "mask": []} for o in OFFSETS}
kept, failed, quality_rows = [], [], []
replay_raw, replay_src, replay_ids = [], [], []
for rec, ev in tqdm(list(zip(df.record, df.event))):
    try:
        raw, sources = to_slot_array(load_segment(RAW / "waveforms" / rec / ev, cfg), cfg)
        outs = {o: preprocess_slots(raw, c) for o, c in cfgs.items()}
    except Exception as e:  # report, don't hide
        failed.append((ev, repr(e))); continue
    for o, pw in outs.items():
        W[o]["signals"].append(pw.signals); W[o]["mask"].append(pw.mask)
    kept.append(ev)
    quality_rows.append({s: q.level for s, q in outs[OFFSETS[0]].quality.items()})
    if ev in sample_ids:
        replay_raw.append(raw); replay_src.append(sources); replay_ids.append(ev)

print(f"kept {len(kept)} / {len(df)}; failed {len(failed)}")
for ev, err in failed[:10]:
    print("  FAILED", ev, err)
assert len(failed) <= 0.005 * len(df), "too many failures - fix the cause (see above) before continuing"

In [ ]:
dk = df.set_index("event").loc[kept].reset_index()
q = pd.DataFrame(quality_rows)
print("slot quality levels over all events (offset", OFFSETS[0], "):")
print(q.apply(lambda c: c.value_counts()).fillna(0).astype(int))

for o in OFFSETS:
    signals = np.stack(W[o]["signals"]); mask = np.stack(W[o]["mask"])
    assert signals.shape == (len(dk), 4, cfgs[o].window_samples) and np.isfinite(signals).all()
    path = DRIVE / "processed" / f"windows_off{o}.npz"
    save_windows(path, signals, mask, dk.y.to_numpy(), dk.split.to_numpy(), dk.event.to_numpy(),
                 dk.record.to_numpy(), cfgs[o])
    print(path.name, signals.shape, f"{path.stat().st_size/1e6:.0f} MB", "| mask means:", mask.mean(0).round(3))

In [ ]:
# Replay sample for the laptop: TEST events only, raw slot-ordered segments.
lab = dk.set_index("event")
y_s = lab.loc[replay_ids, "y"].to_numpy(); pat_s = lab.loc[replay_ids, "record"].to_numpy()
assert (lab.loc[replay_ids, "split"] == "test").all()
save_replay(DRIVE / "replay" / "replay_sample.npz", np.stack(replay_raw), y_s, replay_ids, pat_s, replay_src, cfgs[OFFSETS[0]])
print(len(replay_ids), "replay alarms,", int(y_s.sum()), "true | slot availability:",
      {s: sum(src[s] is not None for src in replay_src) for s in cfg.channel_slots})

(DRIVE / "processed" / "windows_meta.json").write_text(json.dumps(dict(
    n_events=len(dk), n_failed=len(failed), seed=SEED, offsets=OFFSETS, vtac_ref=GIT_REF,
    split_counts=dk.split.value_counts().to_dict(), true_rate_by_split=dk.groupby("split").y.mean().round(4).to_dict(),
    val_carved=bool(not ({"val", "valid", "validation"} & set(split.split.str.lower()))),
    dataset="PhysioNet VTaC 1.0"), indent=2))

## 5. Sanity plots: 3 true + 3 false (preprocessed window, offset 2.0)

In [ ]:
import matplotlib.pyplot as plt
S = np.stack(W[OFFSETS[0]]["signals"]); Mk = np.stack(W[OFFSETS[0]]["mask"])
pick = list(np.flatnonzero(dk.y == 1)[:3]) + list(np.flatnonzero(dk.y == 0)[:3])
t = np.arange(cfg.window_samples) / cfg.fs
fig, axes = plt.subplots(len(pick), 1, figsize=(12, 2.2 * len(pick)), sharex=True)
for ax, i in zip(axes, pick):
    for c, slot in enumerate(cfg.channel_slots):
        if Mk[i, c]:
            ax.plot(t, S[i, c] + 4 * c, lw=0.7, label=slot)
    ax.set_ylabel(("TRUE " if dk.y[i] else "false ") + dk.event[i][:8], fontsize=7)
axes[0].legend(ncol=4, fontsize=7); plt.tight_layout()
plt.savefig(DRIVE / "reports" / "window_samples.png", dpi=110); plt.show()

## Done - next steps

1. Paste the printed survey/split numbers into `docs/data.md` -> Findings (replace the "to be
   confirmed" rows) and commit.
2. Run `02_baseline` and `03_cnn` (both read `processed/windows_off*.npz`).
3. Download `replay/replay_sample.npz` + `replay/replay_index.json` into the repo's `data/sample/`.